# 🗑️ SmartWaste AI — Phase 1: Dataset Setup & Verification

**Run this notebook on Google Colab with GPU enabled.**

This notebook:
1. Clones / mounts the SmartWaste AI repository
2. Installs all dependencies
3. Downloads and prepares the TrashNet dataset
4. Verifies dataset integrity
5. Checks GPU availability

---
**TrashNet License**: MIT — Gary Thung & Mindy Yang (Stanford)

**Classes**: cardboard, glass, metal, paper, plastic, trash (6 total)


## Cell 1 — Mount Google Drive (optional but recommended)

Mount Drive to persist the dataset between Colab sessions so you don't re-download every time.


In [ ]:
# Optional: mount Google Drive to persist dataset
MOUNT_DRIVE = True   # Set to False if you don't want to use Drive

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    print('Google Drive mounted at /content/drive')
else:
    print('Skipping Drive mount — dataset will be lost when session ends.')


## Cell 2 — Clone or Upload Repository


In [ ]:
import os

# ── Option A: Clone from GitHub (replace with your repo URL) ──────────────
# REPO_URL = 'https://github.com/YOUR_USERNAME/SmartWaste_Ai.git'
# !git clone {REPO_URL} /content/SmartWaste_Ai
# os.chdir('/content/SmartWaste_Ai')

# ── Option B: If you uploaded a ZIP to Drive ──────────────────────────────
# !unzip /content/drive/MyDrive/SmartWaste_Ai.zip -d /content/
# os.chdir('/content/SmartWaste_Ai')

# ── Option C: Create the structure directly in Colab ─────────────────────
PROJECT_ROOT = '/content/SmartWaste_Ai'
os.makedirs(PROJECT_ROOT, exist_ok=True)
os.chdir(PROJECT_ROOT)
print(f'Working directory: {os.getcwd()}')


## Cell 3 — Install Dependencies


In [ ]:
# Install PyTorch with CUDA 12.1 (Colab T4/A100 default)
!pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121 -q

# Install remaining dependencies
!pip install \
    fastapi uvicorn[standard] pydantic pydantic-settings python-multipart \
    sqlalchemy aiosqlite httpx \
    scikit-learn matplotlib seaborn pillow tqdm \
    gdown requests \
    grad-cam \
    -q

print('All dependencies installed.')


## Cell 4 — Verify GPU


In [ ]:
import torch

print(f'PyTorch version  : {torch.__version__}')
print(f'CUDA available   : {torch.cuda.is_available()}')

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU              : {gpu}')
    print(f'VRAM             : {vram:.1f} GB')
else:
    print('WARNING: No GPU detected. Go to Runtime > Change runtime type > T4 GPU')


## Cell 5 — Set Up Project Structure


In [ ]:
import sys
sys.path.insert(0, '/content/SmartWaste_Ai')

# Create all required directories
dirs = [
    'dataset/raw', 'dataset/processed/train', 'dataset/processed/val',
    'dataset/processed/test', 'ml', 'models',
    'results/graphs', 'results/confusion_matrix', 'results/reports',
    'notebooks', 'tests'
]
import os
for d in dirs:
    os.makedirs(f'/content/SmartWaste_Ai/{d}', exist_ok=True)

print('Directory structure ready.')


## Cell 6 — Download TrashNet Dataset

Downloads ~150 MB from Google Drive (official source).
If this fails, a HuggingFace fallback is used automatically.


In [ ]:
# Run the download script
!python /content/SmartWaste_Ai/dataset/download_trashnet.py


## Cell 7 — Verify Dataset Integrity


In [ ]:
!python /content/SmartWaste_Ai/dataset/verify_dataset.py


## Cell 8 — Visualise Sample Images Per Class


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
from PIL import Image
import random

PROCESSED = Path('/content/SmartWaste_Ai/dataset/processed/train')
CLASSES = ['cardboard', 'glass', 'metal', 'paper', 'plastic', 'trash']
SAMPLES_PER_CLASS = 4
random.seed(42)

fig, axes = plt.subplots(
    len(CLASSES), SAMPLES_PER_CLASS,
    figsize=(SAMPLES_PER_CLASS * 3, len(CLASSES) * 3)
)
fig.suptitle('TrashNet — Sample Images Per Class', fontsize=16, y=1.01)

for row, class_name in enumerate(CLASSES):
    class_dir = PROCESSED / class_name
    images = sorted(p for p in class_dir.iterdir() if p.suffix.lower() in {'.jpg', '.jpeg', '.png'})
    samples = random.sample(images, min(SAMPLES_PER_CLASS, len(images)))

    for col, img_path in enumerate(samples):
        ax = axes[row][col]
        img = Image.open(img_path).convert('RGB')
        ax.imshow(img)
        ax.axis('off')
        if col == 0:
            ax.set_ylabel(class_name.upper(), rotation=90, fontsize=11, labelpad=10)

plt.tight_layout()
plt.savefig('/content/SmartWaste_Ai/results/graphs/sample_images.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved → results/graphs/sample_images.png')


## Cell 9 — Class Distribution Bar Chart


In [ ]:
import json
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

PROCESSED = Path('/content/SmartWaste_Ai/dataset/processed')
CLASSES = ['cardboard', 'glass', 'metal', 'paper', 'plastic', 'trash']
SPLITS = ['train', 'val', 'test']
COLORS = ['#4A90D9', '#50C878', '#FF6B6B']

counts = {split: [] for split in SPLITS}
for split in SPLITS:
    for cls in CLASSES:
        d = PROCESSED / split / cls
        n = len(list(d.glob('*.[jp][pn][g]*'))) if d.exists() else 0
        counts[split].append(n)

x = np.arange(len(CLASSES))
width = 0.25

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Stacked bar chart ──────────────────────────────────────────────────────
ax1 = axes[0]
bottoms = np.zeros(len(CLASSES))
for i, (split, color) in enumerate(zip(SPLITS, COLORS)):
    bars = ax1.bar(x, counts[split], width=0.6, bottom=bottoms, label=split.capitalize(), color=color, alpha=0.9)
    bottoms += np.array(counts[split])

ax1.set_title('Image Count per Class (Stacked by Split)', fontsize=13)
ax1.set_xticks(x)
ax1.set_xticklabels([c.capitalize() for c in CLASSES], rotation=15)
ax1.set_ylabel('Number of Images')
ax1.legend()
ax1.grid(axis='y', alpha=0.3)

# ── Pie chart of total distribution ───────────────────────────────────────
ax2 = axes[1]
totals = [sum(counts[s][i] for s in SPLITS) for i in range(len(CLASSES))]
PALETTE = ['#E74C3C','#3498DB','#2ECC71','#F39C12','#9B59B6','#1ABC9C']
wedges, texts, autotexts = ax2.pie(
    totals, labels=[c.capitalize() for c in CLASSES],
    autopct='%1.1f%%', colors=PALETTE, startangle=140,
    pctdistance=0.82, labeldistance=1.08
)
for at in autotexts:
    at.set_fontsize(9)
ax2.set_title('Total Class Distribution (All Splits)', fontsize=13)

plt.suptitle('TrashNet Dataset — Class Distribution Analysis', fontsize=15, y=1.02)
plt.tight_layout()
plt.savefig('/content/SmartWaste_Ai/results/graphs/class_distribution.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved → results/graphs/class_distribution.png')

print('\n⚠  Class Imbalance Note:')
for cls, total in zip(CLASSES, totals):
    pct = 100.0 * total / sum(totals)
    bar = '█' * int(pct / 2)
    print(f'  {cls:<12} {total:>4} images  ({pct:>5.1f}%)  {bar}')


## Cell 10 — Compute Class Weights for Imbalance Handling

We compute class weights using **inverse frequency**.  
These will be passed to `CrossEntropyLoss(weight=...)` during training.


In [ ]:
import torch
import numpy as np
from pathlib import Path

PROCESSED = Path('/content/SmartWaste_Ai/dataset/processed/train')
CLASSES = ['cardboard', 'glass', 'metal', 'paper', 'plastic', 'trash']

# Count training images per class
train_counts = []
for cls in CLASSES:
    d = PROCESSED / cls
    n = len([p for p in d.glob('*') if p.suffix.lower() in {'.jpg','.jpeg','.png'}]) if d.exists() else 0
    train_counts.append(n)

total_train = sum(train_counts)
num_classes = len(CLASSES)

# Inverse frequency weights
# weight[c] = total / (num_classes * count[c])
# Higher weight → model penalised more for misclassifying rare classes
weights = [total_train / (num_classes * max(n, 1)) for n in train_counts]
class_weights = torch.tensor(weights, dtype=torch.float32)

print('Training class weights (for CrossEntropyLoss):')
print('=' * 45)
for i, (cls, w, n) in enumerate(zip(CLASSES, weights, train_counts)):
    bar = '█' * int(w * 10)
    print(f'  [{i}] {cls:<12}  n={n:>4}  weight={w:.4f}  {bar}')
print('=' * 45)
print(f'\nclass_weights tensor: {class_weights}')
print(f'\nNote: trash has the highest weight ({max(weights):.4f})')
print('      because it has the fewest training samples.')

# Save for use in training scripts
import json
weight_dict = dict(zip(CLASSES, [round(w, 6) for w in weights]))
out_path = '/content/SmartWaste_Ai/results/reports/class_weights.json'
with open(out_path, 'w') as f:
    json.dump(weight_dict, f, indent=2)
print(f'\nClass weights saved → {out_path}')


## ✅ Phase 1 Complete

**What was accomplished:**
- TrashNet dataset downloaded and extracted
- Stratified 70/15/15 split applied with seed=42
- 5-point integrity verification passed
- Class distribution visualised
- Class weights computed for imbalance handling

**Next: Phase 2 — Preprocessing + Custom Dataset Class**  
Open `notebooks/02_preprocessing_and_training.ipynb`
